In [20]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

panel = pd.read_csv("../data/clean_data/zh_regression_panel.csv")

In [21]:
df = panel[panel["year"] >= 2011]
df = df[df["year"] <= 2022]
df = df.drop(columns=["tertiary_share"])
df = df.dropna()

df["log_median_income"] = np.log(df["median_income"])
df["log_population_density"] = np.log(df["population_density"])


In [22]:
print(df.shape)

(1677, 11)


In [23]:
formula = "turnout ~ foreigner_rate + social_assistance_rate + unemployment_rate + log_median_income + old_age_ratio + log_population_density"
#reg layout

In [24]:
factors = ["foreigner_rate", "social_assistance_rate", "unemployment_rate",
           "log_median_income", "old_age_ratio", "log_population_density"]

In [25]:
model1 = smf.ols(formula, data=df).fit(cov_type="cluster", cov_kwds={"groups": df["GDENR"]})
print(model1.summary())
print(model1.params[factors].round(3))
print(model1.pvalues[factors].round(3))
#pooled OLS which disregards fixed effects but runs on panel

                            OLS Regression Results                            
Dep. Variable:                turnout   R-squared:                       0.454
Model:                            OLS   Adj. R-squared:                  0.452
Method:                 Least Squares   F-statistic:                     107.8
Date:                Mon, 23 Nov 2026   Prob (F-statistic):           4.28e-53
Time:                        18:52:09   Log-Likelihood:                -5328.0
No. Observations:                1677   AIC:                         1.067e+04
Df Residuals:                    1670   BIC:                         1.071e+04
Df Model:                           6                                         
Covariance Type:              cluster                                         
                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept               -185

In [26]:
model2 = smf.ols(formula + " + C(GDENR)", data=df).fit(cov_type="cluster", cov_kwds={"groups": df["GDENR"]})


print(model2.params[factors])
print(model2.pvalues[factors])
#just print the factors coefficients, otherwise would also the municipality constants.

foreigner_rate            -0.109602
social_assistance_rate    -1.236880
unemployment_rate          4.509246
log_median_income         37.662838
old_age_ratio              0.348973
log_population_density     0.455520
dtype: float64
foreigner_rate            3.679507e-01
social_assistance_rate    7.873085e-06
unemployment_rate         8.819455e-52
log_median_income         4.711326e-08
old_age_ratio             7.787599e-10
log_population_density    9.457651e-01
dtype: float64


In [27]:
model3 = smf.ols(formula + " + C(GDENR) + C(year)", data=df).fit(
    cov_type="cluster", cov_kwds={"groups": df["GDENR"]})
#including year fixed effect

print(model3.params[factors].round(3))
print(model3.pvalues[factors].round(3))

foreigner_rate           -0.124
social_assistance_rate   -0.014
unemployment_rate        -0.015
log_median_income         9.412
old_age_ratio             0.109
log_population_density   -7.137
dtype: float64
foreigner_rate            0.079
social_assistance_rate    0.922
unemployment_rate         0.939
log_median_income         0.000
old_age_ratio             0.009
log_population_density    0.017
dtype: float64


In [28]:
results = pd.DataFrame({
    "Model 1 (OLS)": model1.params[factors],
    "Model 2 (+ municipality FE)": model2.params[factors],
    "Model 3 (+ year FE)": model3.params[factors],
})
print(results.round(2))

                        Model 1 (OLS)  Model 2 (+ municipality FE)  \
foreigner_rate                  -0.31                        -0.11   
social_assistance_rate          -0.76                        -1.24   
unemployment_rate                2.49                         4.51   
log_median_income               21.43                        37.66   
old_age_ratio                    0.39                         0.35   
log_population_density          -1.03                         0.46   

                        Model 3 (+ year FE)  
foreigner_rate                        -0.12  
social_assistance_rate                -0.01  
unemployment_rate                     -0.02  
log_median_income                      9.41  
old_age_ratio                          0.11  
log_population_density                -7.14  


In [29]:
#model using longer period
df_long = panel[panel["year"] >= 1999]
df_long = df_long[df_long["year"] <= 2022]
df_long = df_long[["GDENR", "year", "turnout", "foreigner_rate", "median_income",
                   "old_age_ratio", "population_density"]]
df_long = df_long.dropna()

In [30]:
df_long["log_median_income"] = np.log(df_long["median_income"])
df_long["log_population_density"] = np.log(df_long["population_density"])

In [31]:
factors_long = ["foreigner_rate", "log_median_income", "old_age_ratio", "log_population_density"]
formula_long = "turnout ~ foreigner_rate + log_median_income + old_age_ratio + log_population_density"

In [32]:
model4 = smf.ols(formula_long + " + C(GDENR) + C(year)", data=df_long).fit(cov_type="cluster", cov_kwds={"groups": df_long["GDENR"]})

In [33]:
print(df_long.shape)
print(model4.params[factors_long].round(3))
print(model4.pvalues[factors_long].round(3))

(3757, 9)
foreigner_rate            0.032
log_median_income         7.705
old_age_ratio             0.171
log_population_density   -5.942
dtype: float64
foreigner_rate            0.563
log_median_income         0.000
old_age_ratio             0.000
log_population_density    0.001
dtype: float64


In [34]:
df5 = panel[panel["year"] >= 2017]
df5 = df5[df5["year"] <= 2022]
df5 = df5.dropna()
df5["log_median_income"] = np.log(df5["median_income"])
df5["log_population_density"] = np.log(df5["population_density"])
#make data ready for model with tertiary education

In [35]:
model5 = smf.ols(formula + " + tertiary_share + C(GDENR) + C(year)", data=df5).fit(cov_type="cluster", cov_kwds={"groups": df5["GDENR"]})
print(model5.params[factors + ["tertiary_share"]].round(3))
print(model5.pvalues[factors + ["tertiary_share"]].round(3))
print(df5.shape)
#get model with tertiary share

foreigner_rate           -0.053
social_assistance_rate    0.218
unemployment_rate        -0.370
log_median_income         3.213
old_age_ratio             0.341
log_population_density    0.380
tertiary_share            0.208
dtype: float64
foreigner_rate            0.660
social_assistance_rate    0.314
unemployment_rate         0.150
log_median_income         0.410
old_age_ratio             0.000
log_population_density    0.916
tertiary_share            0.013
dtype: float64
(844, 12)


In [36]:
df6 = df[df["GDENR"] < 291]

In [37]:
model6 = smf.ols(formula + " + C(GDENR) + C(year)", data=df6).fit(cov_type="cluster", cov_kwds={"groups": df6["GDENR"]})
#model without the merged municipalities

In [38]:
print(model6.params[factors].round(3))
print(model6.pvalues[factors].round(3))
print(df6.shape)

foreigner_rate           -0.123
social_assistance_rate   -0.002
unemployment_rate        -0.012
log_median_income         9.387
old_age_ratio             0.109
log_population_density   -7.147
dtype: float64
foreigner_rate            0.083
social_assistance_rate    0.987
unemployment_rate         0.954
log_median_income         0.000
old_age_ratio             0.010
log_population_density    0.017
dtype: float64
(1635, 11)
